# 01. Exploratory Data Analysis (EDA)
### Hotel Demand Forecasting and Cancellation Prediction Using Apache Spark

This notebook performs in-depth exploratory data analysis on the Antonio et al. (2019) Hotel Booking Demand dataset (~119k records).
Key research questions investigated:
1. What is the monthly seasonality of bookings across City Hotel and Resort Hotel?
2. How do cancellation rates differ between properties?
3. What is the relationship between booking lead time and cancellations?
4. How do distribution channels and market segments perform?
5. What explains the "Deposit Paradox" (extreme cancellations in non-refundable bookings)?
6. How does Average Daily Rate (ADR) vary seasonally?


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root
sys.path.insert(0, str(Path().resolve().parent))
from src.cleaning import clean_bookings_pandas

# Load and clean dataset
raw_df = pd.read_csv('../hotel_bookings.csv')
df = clean_bookings_pandas(raw_df)
print(f"Total cleaned bookings: {len(df):,}")
df.head()


## 1. Property-Level Cancellation Rates
City Hotel exhibits a substantially higher cancellation frequency (~41.7%) than Resort Hotel (~27.8%).


In [ ]:
hotel_canc = df.groupby('hotel')['is_canceled'].agg(['count', 'mean']).reset_index()
hotel_canc['cancel_pct'] = hotel_canc['mean'] * 100
hotel_canc


## 2. Lead Time Distribution
Bookings made more than 90 days in advance exhibit an escalating risk of cancellation.


In [ ]:
plt.figure(figsize=(9, 4.5))
sns.kdeplot(data=df[df['is_canceled']==0]['lead_time'], label='Confirmed (0)', fill=True, color='green')
sns.kdeplot(data=df[df['is_canceled']==1]['lead_time'], label='Canceled (1)', fill=True, color='red')
plt.title('Lead Time Distribution: Confirmed vs Canceled')
plt.xlabel('Lead Time (Days)')
plt.legend()
plt.show()


## 3. The Deposit Paradox
Why do Non-Refundable deposits have >99% cancellation rates?
Hotels systematically impose non-refundable requirements on high-risk group reservations and tour operators that later default or cancel in bulk.


In [ ]:
df.groupby('deposit_type')['is_canceled'].agg(['count', 'mean'])


## 4. Monthly ADR Seasonality
Resort Hotel experiences an aggressive summer rate spike (€150+/night in July/August), while City Hotel maintains stable pricing year-round.


In [ ]:
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
monthly_adr = df.groupby(['arrival_date_month', 'hotel'])['adr'].mean().unstack().reindex(month_order)
monthly_adr.plot(figsize=(10, 4.5), marker='o')
plt.title('Monthly ADR by Hotel Type (€)')
plt.ylabel('Average Daily Rate (€)')
plt.show()
